
# v4 Validation: Viable Configurations vs Market Option Prices

This notebook validates v4 viable-region configurations by comparing simulated corridor premium levels to **comparable real-world option prices** (put-spread proxy) from available market data.

Validation flow:
1. Load viable region points from `v4_viability_region_points.csv`.
2. Select representative configs from the viable set.
3. Recompute corridor premiums under historical paths.
4. Build market put-spread proxy from live/cached option chain (Lyra and other collected venues).
5. Compare corridor premium distribution vs market put-spread cost distribution.


In [1]:

import os, glob, time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from numpy.polynomial.hermite import hermgauss
from datetime import datetime, timezone
import requests

plt.style.use('seaborn-v0_8-whitegrid')
rng = np.random.default_rng(42)

NOTEBOOK_DATA_DIR = 'data'
DATA_DIR_CANDIDATES = [
    NOTEBOOK_DATA_DIR,
    os.path.join('lh-protocol-v3', 'notebooks', 'data'),
    os.path.join('notebooks', 'data'),
]
ROOT_DATA_DIR = None
for _p in DATA_DIR_CANDIDATES:
    if os.path.exists(_p):
        ROOT_DATA_DIR = _p
        break
if ROOT_DATA_DIR is None:
    ROOT_DATA_DIR = NOTEBOOK_DATA_DIR
os.makedirs(ROOT_DATA_DIR, exist_ok=True)

T_WEEK = 7/365
WIDTH = 0.075
N_LIQ = 10_000
BARRIER_BPS = 750

VOL_MARKUP_FLOOR = 0.70
VOL_MARKUP_CAP = 1.80
AMM_MULT_FLOOR = 0.75
AMM_MULT_CAP = 1.60

print('Environment ready.')


Environment ready.


In [2]:

# --- Load viable region configurations ---
viab_path = os.path.join(ROOT_DATA_DIR, 'v4_viability_region_points.csv')
if not os.path.exists(viab_path):
    raise FileNotFoundError(f'Missing viability points file: {viab_path}')

region = pd.read_csv(viab_path)
if 'viable' not in region.columns:
    raise ValueError('Expected `viable` column in viability region file.')

region['viable'] = region['viable'].astype(bool)
viable = region[region['viable']].copy()
print(f'Total points: {len(region)} | viable points: {len(viable)}')
if len(viable) == 0:
    raise ValueError('No viable points found.')

# Representative picks: LP-leaning, RT-leaning, balanced, plus two random viable points.
picks = []
vi = viable.copy()
vi['balance_score'] = (vi['lp_mean_%'] - vi['rt_mean_%']).abs()

p_lp = vi.sort_values(['lp_mean_%', 'rt_mean_%'], ascending=False).head(1)
p_rt = vi.sort_values(['rt_mean_%', 'lp_mean_%'], ascending=False).head(1)
p_bal = vi.sort_values(['balance_score', 'lp_mean_%'], ascending=[True, False]).head(1)

picks.extend([p_lp, p_rt, p_bal])
if len(vi) > 5:
    picks.append(vi.sample(2, random_state=42))

cfg = pd.concat(picks).drop_duplicates()
cfg = cfg.reset_index(drop=True)
cfg['config_id'] = [f'cfg_{i+1}' for i in range(len(cfg))]

show_cols = ['config_id','alpha_rv','alpha_ivrv','k_lin','k_quad','yield_share_to_rt','max_utilization','lp_mean_%','rt_mean_%']
print(cfg[show_cols].to_string(index=False, float_format=lambda x: f'{x:,.3f}'))


Total points: 1728 | viable points: 26
config_id  alpha_rv  alpha_ivrv  k_lin  k_quad  yield_share_to_rt  max_utilization  lp_mean_%  rt_mean_%
    cfg_1     0.100       0.200  0.260   0.100              0.050            0.600      0.235      0.146
    cfg_2     0.180       0.200  0.100   0.100              0.080            0.800      0.005      0.153
    cfg_3     0.250       0.500  0.100   0.220              0.050            0.700      0.154      0.150
    cfg_4     0.100       0.300  0.260   0.220              0.050            0.600      0.053      0.147
    cfg_5     0.180       0.200  0.260   0.100              0.050            0.800      0.085      0.140


In [3]:

# --- Price-path utilities (same structural model as v4) ---

def fetch_birdeye_closes(days_back=730):
    key='ed577a4a6a4f480fa659b4f18673e4b1'
    mint='So11111111111111111111111111111111111111112'
    url='https://public-api.birdeye.so/defi/ohlcv'
    headers={'X-API-KEY':key,'Accept':'application/json'}
    now=int(time.time())
    t=now-days_back*86400
    out=[]
    while t<now:
        t2=min(t+90*86400,now)
        try:
            js=requests.get(url,headers=headers,params={'address':mint,'type':'1D','time_from':t,'time_to':t2},timeout=10).json()
            out.extend(js.get('data',{}).get('items',[]))
        except Exception:
            pass
        t=t2
    d={}
    for it in out:
        ts=it.get('unixTime',0)
        c=float(it.get('c',0) or 0)
        if c>0: d[ts]=c
    arr=sorted(d.items())
    return np.array([x[1] for x in arr],dtype=float)

closes = fetch_birdeye_closes()
if len(closes) == 0:
    # fallback synthetic
    z = rng.standard_normal(730)
    sig = 0.70/np.sqrt(365)
    closes = 100*np.exp(np.cumsum((-0.5*sig*sig)+sig*z))


def rolling_vol(prices, window):
    lr=np.diff(np.log(prices))
    out=np.full(len(prices),np.nan)
    for i in range(window, len(lr)+1):
        out[i]=float(np.std(lr[i-window:i], ddof=1)*np.sqrt(365))
    return out

vol7 = rolling_vol(closes, 7)
vol30 = rolling_vol(closes, 30)


def cl_value_vec(S,L,pl,pu):
    S=np.atleast_1d(np.asarray(S,float))
    spl,spu=np.sqrt(pl),np.sqrt(pu)
    sp=np.sqrt(np.clip(S,1e-10,None))
    bl=S<=pl; ab=S>=pu
    a=np.where(bl, L*(spu-spl)/(spl*spu), np.where(ab,0.0,L*(spu-sp)/(sp*spu)))
    b=np.where(bl, 0.0, np.where(ab, L*(spu-spl), L*(sp-spl)))
    return a*S+b


def corridor_payoff_vec(ST,S0,B,Cap,L,pl,pu):
    V0=cl_value_vec(S0,L,pl,pu)
    Ve=cl_value_vec(np.maximum(ST,B),L,pl,pu)
    return np.minimum(Cap, np.maximum(0.0, np.where(ST>=S0, 0.0, V0-Ve)))


def fv_quadrature(S0,sigma,B,Cap,L,pl,pu,T=T_WEEK):
    n,w=hermgauss(64)
    ST=S0*np.exp(-sigma*sigma/2*T + sigma*np.sqrt(T)*n*np.sqrt(2))
    pay=corridor_payoff_vec(ST,S0,B,Cap,L,pl,pu)
    return max(0.0, float(np.sum(w*pay)/np.sqrt(np.pi)))


def make_pos(S0,lev=1.0):
    pl=S0*(1-WIDTH); pu=S0*(1+WIDTH)
    B=max(S0*(1-BARRIER_BPS/10_000),pl)
    V0=float(cl_value_vec(np.array([S0]),N_LIQ*lev,pl,pu)[0])
    Vb=float(cl_value_vec(np.array([B]),N_LIQ*lev,pl,pu)[0])
    return {'pl':pl,'pu':pu,'B':B,'V0':V0,'cap':max(0,V0-Vb),'L':N_LIQ*lev}


def vol_markup(rv30,ivrv,alpha_rv,alpha_ivrv,rv_ref=0.60):
    return float(np.clip(1 + alpha_rv*(rv30/max(rv_ref,1e-9)-1) + alpha_ivrv*(ivrv-1), VOL_MARKUP_FLOOR, VOL_MARKUP_CAP))


def amm_mult(dem,sup,k_lin,k_quad):
    if sup<=0:
        return AMM_MULT_CAP
    im=dem/sup - 1.0
    return float(np.clip(1 + k_lin*im + k_quad*np.sign(im)*(im**2), AMM_MULT_FLOOR, AMM_MULT_CAP))

print(f'Price paths ready: {len(closes)} daily points')


Price paths ready: 730 daily points


In [4]:

# --- Corridor premium distribution per selected viable config ---

def corridor_premium_distribution(params, n_weeks_limit=99):
    alpha_rv = float(params['alpha_rv'])
    alpha_ivrv = float(params['alpha_ivrv'])
    k_lin = float(params['k_lin'])
    k_quad = float(params['k_quad'])
    util = float(params['max_utilization'])

    rt_cap = 2_000_000.0
    prem = []
    fair = []
    cap = []

    n_weeks = 0
    for si in range(35, len(closes)-7, 7):
        ei = si+7
        Ss = float(closes[si])
        rv30 = float(vol30[si]) if np.isfinite(vol30[si]) else 0.65
        rv7 = float(vol7[si]) if np.isfinite(vol7[si]) else rv30
        ivrv = float(np.clip(0.95 + 0.45*(rv7/max(rv30,1e-9)), 0.90, 1.45))

        # use representative leverage 1.0 for pricing comparison layer
        pos = make_pos(Ss, lev=1.0)
        fv = fv_quadrature(Ss, rv30, pos['B'], pos['cap'], pos['L'], pos['pl'], pos['pu'])

        # approximate weekly demand from same Poisson mechanism mean
        dem = max(1.0, 4.0*np.clip(rv7/max(rv30,1e-9), 0.6, 1.8)) * pos['cap']
        sup = max(1.0, rt_cap * util)

        mv = vol_markup(rv30, ivrv, alpha_rv, alpha_ivrv)
        ma = amm_mult(dem, sup, k_lin, k_quad)

        p = fv * mv * ma
        prem.append(float(p))
        fair.append(float(fv))
        cap.append(float(pos['cap']))

        # soft capital update (pricing-only projection)
        rt_cap = max(1.0, rt_cap * (1.0 + 0.001 + (p - fv) / max(rt_cap,1e-9)))
        n_weeks += 1
        if n_weeks >= n_weeks_limit:
            break

    return np.array(prem), np.array(fair), np.array(cap)

rows=[]
for _,r in cfg.iterrows():
    p,f,c = corridor_premium_distribution(r)
    rows.append({
        'config_id': r['config_id'],
        'corr_prem_med': float(np.median(p)),
        'corr_prem_mean': float(np.mean(p)),
        'corr_cap_med': float(np.median(c)),
        'corr_prem_per_cap_med': float(np.median(p/np.maximum(c,1e-9))),
        'corr_prem_per_cap_mean': float(np.mean(p/np.maximum(c,1e-9))),
        'corr_over_fv_med': float(np.median(p/np.maximum(f,1e-9))),
        'corr_over_fv_mean': float(np.mean(p/np.maximum(f,1e-9))),
    })

corr_df = pd.DataFrame(rows)
print(corr_df.to_string(index=False, float_format=lambda x: f'{x:,.3f}'))


config_id  corr_prem_med  corr_prem_mean  corr_cap_med  corr_prem_per_cap_med  corr_prem_per_cap_mean  corr_over_fv_med  corr_over_fv_mean
    cfg_1        157.631         157.427       510.829                  0.302                   0.304             0.824              0.828
    cfg_2        171.140         172.399       510.829                  0.328                   0.333             0.898              0.904
    cfg_3        176.545         180.151       510.829                  0.343                   0.348             0.935              0.944
    cfg_4        160.889         162.426       510.829                  0.312                   0.314             0.853              0.854
    cfg_5        160.364         161.549       510.829                  0.308                   0.312             0.842              0.847


In [5]:

# --- Market option comparator (put spread proxy from available option chain) ---
# Uses latest cached multivenue file produced earlier.
opt_files = sorted(glob.glob(os.path.join(ROOT_DATA_DIR, 'sol_options_multivenue_*.csv')))
if not opt_files:
    raise FileNotFoundError('No option market dataset found in notebooks/data.')

opt_path = opt_files[-1]
opt = pd.read_csv(opt_path)
for c in ['mark','strike','spot','T_years']:
    opt[c] = pd.to_numeric(opt[c], errors='coerce')

opt = opt.dropna(subset=['symbol','mark','strike','spot','T_years','option_type'])
opt = opt[(opt['mark']>0) & (opt['spot']>0) & (opt['T_years']>0)]

# Keep put options near 1-week tenor as comparator.
opt_put = opt[opt['option_type'].astype(str).str.lower()=='put'].copy()
opt_put = opt_put[(opt_put['T_years']*365 >= 2) & (opt_put['T_years']*365 <= 21)]

spread_costs=[]
for (venue, expiry), g in opt_put.groupby(['venue','expiry_utc']):
    if len(g)<4:
        continue
    spot=float(np.median(g['spot']))
    k_atm = spot
    k_otm = spot*(1-WIDTH)

    # nearest strikes by absolute distance
    g = g.copy()
    g['d_atm'] = (g['strike'] - k_atm).abs()
    g['d_otm'] = (g['strike'] - k_otm).abs()

    row_atm = g.sort_values('d_atm').iloc[0]
    row_otm = g.sort_values('d_otm').iloc[0]

    c_atm = float(row_atm['mark'])
    c_otm = float(row_otm['mark'])
    spread = max(0.0, c_atm - c_otm)

    cap_unit = max(1e-9, float(row_atm['strike']) - float(row_otm['strike']))
    if spread > 0:
        spread_costs.append({
            'venue': venue,
            'expiry_utc': expiry,
            'spot': spot,
            'k_atm': float(row_atm['strike']),
            'k_otm': float(row_otm['strike']),
            'put_spread_cost': spread,
            'put_spread_cap_unit': cap_unit,
            'put_spread_cost_per_cap': spread / cap_unit,
            'T_days': float(np.median(g['T_years'])*365),
        })

mkt_spread_df = pd.DataFrame(spread_costs)
if len(mkt_spread_df)==0:
    raise ValueError('No comparable market put spread costs could be constructed from option dataset.')

print(f'Option market source: {opt_path}')
print(f'Comparable put-spread points: {len(mkt_spread_df)}')
print(mkt_spread_df.head(10).to_string(index=False, float_format=lambda x: f'{x:,.3f}'))

mkt_med = float(np.median(mkt_spread_df['put_spread_cost']))
mkt_mean = float(np.mean(mkt_spread_df['put_spread_cost']))
mkt_per_cap_med = float(np.median(mkt_spread_df['put_spread_cost_per_cap']))
mkt_per_cap_mean = float(np.mean(mkt_spread_df['put_spread_cost_per_cap']))
print(f'\\nMarket put-spread cost median: ${mkt_med:.2f} | mean: ${mkt_mean:.2f}')
print(f'Market put-spread cost per cap-unit median: {mkt_per_cap_med:.4f} | mean: {mkt_per_cap_mean:.4f}')


Option market source: data/sol_options_multivenue_20260414_091910.csv
Comparable put-spread points: 3
venue                expiry_utc   spot  k_atm  k_otm  put_spread_cost  put_spread_cap_unit  put_spread_cost_per_cap  T_days
 lyra 2026-04-17T00:00:00+00:00 86.200 86.000 80.000            1.400                6.000                    0.233   2.612
 lyra 2026-04-24T00:00:00+00:00 86.200 86.000 80.000            1.800                6.000                    0.300   9.612
 lyra 2026-05-01T00:00:00+00:00 86.200 86.000 80.000            2.100                6.000                    0.350  16.612
\nMarket put-spread cost median: $1.80 | mean: $1.77
Market put-spread cost per cap-unit median: 0.3000 | mean: 0.2944


In [6]:

# --- Compare corridor premium vs market option spread ---
comp = corr_df.copy()
comp['mkt_put_spread_med'] = mkt_med
comp['mkt_put_spread_mean'] = mkt_mean
comp['mkt_put_spread_per_cap_med'] = mkt_per_cap_med
comp['mkt_put_spread_per_cap_mean'] = mkt_per_cap_mean

# Raw (non-normalized) comparison.
comp['premium_minus_mkt_med'] = comp['corr_prem_med'] - mkt_med
comp['premium_to_mkt_med_ratio'] = comp['corr_prem_med'] / max(mkt_med, 1e-9)

# Notional-normalized comparison (cost per protected cap unit).
comp['prem_per_cap_minus_mkt'] = comp['corr_prem_per_cap_med'] - mkt_per_cap_med
comp['prem_per_cap_to_mkt_ratio'] = comp['corr_prem_per_cap_med'] / max(mkt_per_cap_med, 1e-9)

# Convert market per-cap cost into corridor-equivalent dollars for each config's median cap.
comp['mkt_equiv_cost_at_corr_cap'] = comp['corr_cap_med'] * mkt_per_cap_med
comp['corr_minus_mkt_equiv_at_cap'] = comp['corr_prem_med'] - comp['mkt_equiv_cost_at_corr_cap']

print(comp.to_string(index=False, float_format=lambda x: f'{x:,.4f}'))

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))

ax[0].bar(comp['config_id'], comp['corr_prem_med'], color='seagreen', alpha=0.85, label='corridor premium median')
ax[0].bar(comp['config_id'], comp['mkt_equiv_cost_at_corr_cap'], color='darkorange', alpha=0.70, label='market-equivalent cost at corridor cap')
ax[0].set_title('Dollar comparison on matched protected cap')
ax[0].set_ylabel('USD')
ax[0].legend(frameon=False)

ax[1].bar(comp['config_id'], comp['prem_per_cap_to_mkt_ratio'], color='slateblue', alpha=0.85)
ax[1].axhline(1.0, color='black', ls='--', lw=1)
ax[1].set_title('Normalized cost ratio: (corr per-cap) / (market per-cap)')
ax[1].set_ylabel('ratio')

plt.tight_layout()
out_png = os.path.join(ROOT_DATA_DIR, 'v4_validation_premium_vs_market.png')
fig.savefig(out_png, dpi=150)

out_csv = os.path.join(ROOT_DATA_DIR, 'v4_validation_premium_vs_market.csv')
comp.to_csv(out_csv, index=False)

print(f'Saved: {out_png}')
print(f'Saved: {out_csv}')


config_id  corr_prem_med  corr_prem_mean  corr_cap_med  corr_prem_per_cap_med  corr_prem_per_cap_mean  corr_over_fv_med  corr_over_fv_mean  mkt_put_spread_med  mkt_put_spread_mean  mkt_put_spread_per_cap_med  mkt_put_spread_per_cap_mean  premium_minus_mkt_med  premium_to_mkt_med_ratio  prem_per_cap_minus_mkt  prem_per_cap_to_mkt_ratio  mkt_equiv_cost_at_corr_cap  corr_minus_mkt_equiv_at_cap
    cfg_1       157.6314        157.4274      510.8293                 0.3016                  0.3042            0.8242             0.8277              1.8000               1.7667                      0.3000                       0.2944               155.8314                   87.5730                  0.0016                     1.0052                    153.2488                       4.3826
    cfg_2       171.1398        172.3987      510.8293                 0.3282                  0.3331            0.8980             0.9043              1.8000               1.7667                      0.3000     

Saved: data/v4_validation_premium_vs_market.png
Saved: data/v4_validation_premium_vs_market.csv


## Confidence Bands for Market Comparison

Estimate uncertainty of normalized market comparator via:
- cross-expiry dispersion
- bootstrap confidence interval on market put-spread per-cap cost
- implied confidence interval for corridor/market normalized ratio.

In [7]:
# Confidence bands on market per-cap cost and normalized ratio.
if len(mkt_spread_df) < 2:
    print('Not enough market points for confidence bands.')
else:
    x = mkt_spread_df['put_spread_cost_per_cap'].to_numpy(dtype=float)

    # Empirical quantile band across available expiries/venues.
    q10, q50, q90 = np.percentile(x, [10, 50, 90])

    # Bootstrap CI for mean market per-cap cost.
    B = 5000
    boot = np.empty(B, dtype=float)
    n = len(x)
    for b in range(B):
        idx = rng.integers(0, n, size=n)
        boot[b] = float(np.mean(x[idx]))
    ci_low, ci_med, ci_high = np.percentile(boot, [5, 50, 95])

    # Convert into ratio band for each config: corridor_per_cap / market_per_cap.
    band_rows = []
    for _, r in comp.iterrows():
        cpc = float(r['corr_prem_per_cap_med'])
        band_rows.append({
            'config_id': r['config_id'],
            'corr_per_cap': cpc,
            'ratio_q10': cpc / max(q90, 1e-9),
            'ratio_q50': cpc / max(q50, 1e-9),
            'ratio_q90': cpc / max(q10, 1e-9),
            'ratio_ci5': cpc / max(ci_high, 1e-9),
            'ratio_ci50': cpc / max(ci_med, 1e-9),
            'ratio_ci95': cpc / max(ci_low, 1e-9),
        })

    band_df = pd.DataFrame(band_rows)

    print('Market per-cap comparator bands:')
    print(f'  Quantiles [q10, q50, q90] = [{q10:.4f}, {q50:.4f}, {q90:.4f}]')
    print(f'  Bootstrap mean CI [5%,50%,95%] = [{ci_low:.4f}, {ci_med:.4f}, {ci_high:.4f}]')
    print('\nNormalized ratio bands (corridor per-cap / market per-cap):')
    print(band_df.to_string(index=False, float_format=lambda y: f'{y:,.4f}'))

    # Plot ratio with confidence bars.
    fig, ax = plt.subplots(figsize=(8, 4.5))
    xloc = np.arange(len(band_df))
    y = band_df['ratio_ci50'].to_numpy()
    yerr_low = y - band_df['ratio_ci5'].to_numpy()
    yerr_high = band_df['ratio_ci95'].to_numpy() - y

    ax.errorbar(xloc, y, yerr=[yerr_low, yerr_high], fmt='o', capsize=4, color='slateblue', label='bootstrap CI (5-95)')
    ax.scatter(xloc, band_df['ratio_q50'].to_numpy(), color='seagreen', label='cross-expiry median ratio')
    ax.axhline(1.0, color='black', ls='--', lw=1)
    ax.set_xticks(xloc)
    ax.set_xticklabels(band_df['config_id'].tolist())
    ax.set_ylabel('corridor/market normalized ratio')
    ax.set_title('Confidence bands for normalized pricing ratio')
    ax.legend(frameon=False)
    plt.tight_layout()

    out_band_png = os.path.join(ROOT_DATA_DIR, 'v4_validation_ratio_confidence_bands.png')
    out_band_csv = os.path.join(ROOT_DATA_DIR, 'v4_validation_ratio_confidence_bands.csv')
    fig.savefig(out_band_png, dpi=150)
    band_df.to_csv(out_band_csv, index=False)
    print(f'\nSaved: {out_band_png}')
    print(f'Saved: {out_band_csv}')

Market per-cap comparator bands:
  Quantiles [q10, q50, q90] = [0.2467, 0.3000, 0.3400]
  Bootstrap mean CI [5%,50%,95%] = [0.2556, 0.2944, 0.3333]

Normalized ratio bands (corridor per-cap / market per-cap):
config_id  corr_per_cap  ratio_q10  ratio_q50  ratio_q90  ratio_ci5  ratio_ci50  ratio_ci95
    cfg_1        0.3016     0.8869     1.0052     1.2225     0.9047      1.0242      1.1800
    cfg_2        0.3282     0.9654     1.0941     1.3306     0.9847      1.1147      1.2844
    cfg_3        0.3427     1.0080     1.1424     1.3894     1.0282      1.1640      1.3411
    cfg_4        0.3115     0.9162     1.0384     1.2629     0.9346      1.0580      1.2190
    cfg_5        0.3076     0.9046     1.0252     1.2469     0.9227      1.0446      1.2035



Saved: data/v4_validation_ratio_confidence_bands.png
Saved: data/v4_validation_ratio_confidence_bands.csv


## Constrained Feasibility Check (Viability + Price Competitiveness)

Constraint set:
- LP viability: `lp_mean_% > 0`
- RT viability: `rt_mean_% > 0`
- Price competitiveness: `corridor_per_cap / market_per_cap <= 1.0`

If no exact feasible point exists, show nearest frontier points (smallest excess over 1.0 with positive LP/RT means).

In [8]:
# Evaluate constrained feasibility on available validated configurations.
# Join viability metrics from viable region with normalized pricing ratios from validation.

vi_cols = ['alpha_rv','alpha_ivrv','k_lin','k_quad','yield_share_to_rt','max_utilization','lp_mean_%','rt_mean_%','viable']
vi_map = viable[vi_cols].copy()

# Map config rows back to viability params.
cfg_map = cfg[['config_id','alpha_rv','alpha_ivrv','k_lin','k_quad','yield_share_to_rt','max_utilization']].copy()
full = cfg_map.merge(vi_map, on=['alpha_rv','alpha_ivrv','k_lin','k_quad','yield_share_to_rt','max_utilization'], how='left')
full = full.merge(band_df[['config_id','ratio_ci5','ratio_ci50','ratio_ci95']], on='config_id', how='left')
full = full.merge(comp[['config_id','prem_per_cap_to_mkt_ratio','prem_per_cap_minus_mkt']], on='config_id', how='left')

full['feasible_point'] = (full['lp_mean_%'] > 0) & (full['rt_mean_%'] > 0) & (full['prem_per_cap_to_mkt_ratio'] <= 1.0)
full['feasible_ci95'] = (full['lp_mean_%'] > 0) & (full['rt_mean_%'] > 0) & (full['ratio_ci95'] <= 1.0)

print('Constrained feasibility results:')
print(full.to_string(index=False, float_format=lambda z: f'{z:,.4f}'))

if full['feasible_point'].any():
    print('\nExact feasible point(s) found under point estimate ratio <= 1.0:')
    print(full[full['feasible_point']].to_string(index=False, float_format=lambda z: f'{z:,.4f}'))
else:
    print('\nNo exact feasible point found with ratio <= 1.0 among validated configs.')
    # Frontier: positive LP/RT means, minimal excess ratio above 1
    frontier = full[(full['lp_mean_%'] > 0) & (full['rt_mean_%'] > 0)].copy()
    if len(frontier) > 0:
        frontier['ratio_excess'] = frontier['prem_per_cap_to_mkt_ratio'] - 1.0
        frontier = frontier.sort_values(['ratio_excess','lp_mean_%','rt_mean_%'], ascending=[True, False, False])
        print('\nNearest frontier points (smallest ratio excess):')
        print(frontier.head(10).to_string(index=False, float_format=lambda z: f'{z:,.4f}'))

if full['feasible_ci95'].any():
    print('\nRobust feasible point(s) found under CI95 <= 1.0:')
    print(full[full['feasible_ci95']].to_string(index=False, float_format=lambda z: f'{z:,.4f}'))
else:
    print('\nNo robust feasible point under CI95 <= 1.0 (conservative test).')

Constrained feasibility results:
config_id  alpha_rv  alpha_ivrv  k_lin  k_quad  yield_share_to_rt  max_utilization  lp_mean_%  rt_mean_%  viable  ratio_ci5  ratio_ci50  ratio_ci95  prem_per_cap_to_mkt_ratio  prem_per_cap_minus_mkt  feasible_point  feasible_ci95
    cfg_1    0.1000      0.2000 0.2600  0.1000             0.0500           0.6000     0.2353     0.1460    True     0.9047      1.0242      1.1800                     1.0052                  0.0016           False          False
    cfg_2    0.1800      0.2000 0.1000  0.1000             0.0800           0.8000     0.0048     0.1528    True     0.9847      1.1147      1.2844                     1.0941                  0.0282           False          False
    cfg_3    0.2500      0.5000 0.1000  0.2200             0.0500           0.7000     0.1541     0.1498    True     1.0282      1.1640      1.3411                     1.1424                  0.0427           False          False
    cfg_4    0.1000      0.3000 0.2600  0.2200 


## Interpretation Guide

- If `premium_to_mkt_med_ratio > 1`, corridor premium is pricier than market put-spread proxy for sampled maturities/strikes.
- If `< 1`, corridor premium is cheaper.
- Use this with viability outcomes: some higher-pricing configs may still be needed for RT solvency.


## Imbalance Regions and Arbitrage-Agent PnL

This section identifies demand/supply imbalance regimes and simulates a basis-arbitrage agent:
- If protocol hedge is overpriced vs market options, agent sells protocol protection and buys options.
- If protocol hedge is underpriced, agent buys protocol hedge and sells options.

PnL includes execution frictions and a hedge-mismatch penalty (to reflect non-perfect payoff matching).

In [9]:
# Build weekly imbalance series and simulate arbitrage-agent PnL.

# Use nearest-frontier config if available, else first config.
cfg_use = full.sort_values('prem_per_cap_to_mkt_ratio').iloc[0]

alpha_rv = float(cfg_use['alpha_rv'])
alpha_ivrv = float(cfg_use['alpha_ivrv'])
k_lin = float(cfg_use['k_lin'])
k_quad = float(cfg_use['k_quad'])
util = float(cfg_use['max_utilization'])

# Arbitrage policy parameters
EDGE_TRIGGER = 0.03          # trade only if edge > 3% vs market per-cap
MAX_TRADE_SHARE = 0.40       # up to 40% of local protected cap notionals
TX_COST_PCT = 0.01           # 1% execution+fees on traded notional
MISMATCH_PENALTY_PCT = 0.02  # 2% penalty for payoff-shape mismatch

rt_cap = 2_000_000.0
rows=[]

for w, si in enumerate(range(35, len(closes)-7, 7)):
    Ss = float(closes[si])
    rv30 = float(vol30[si]) if np.isfinite(vol30[si]) else 0.65
    rv7 = float(vol7[si]) if np.isfinite(vol7[si]) else rv30
    ivrv = float(np.clip(0.95 + 0.45*(rv7/max(rv30,1e-9)), 0.90, 1.45))

    pos = make_pos(Ss, lev=1.0)
    fv = fv_quadrature(Ss, rv30, pos['B'], pos['cap'], pos['L'], pos['pl'], pos['pu'])

    demand_scale = float(np.clip(rv7/max(rv30,1e-9), 0.6, 1.8))
    demand_notional = 4.0 * demand_scale * pos['cap']
    supply_notional = max(1.0, rt_cap * util)

    imbalance = demand_notional / max(supply_notional, 1e-9) - 1.0
    mv = vol_markup(rv30, ivrv, alpha_rv, alpha_ivrv)
    ma = amm_mult(demand_notional, supply_notional, k_lin, k_quad)

    protocol_per_cap = float((fv * mv * ma) / max(pos['cap'], 1e-9))
    market_per_cap = float(mkt_per_cap_med)
    rel_edge = (protocol_per_cap / max(market_per_cap, 1e-9)) - 1.0

    trade_share = min(MAX_TRADE_SHARE, max(0.0, abs(imbalance) * 2.0))
    trade_notional = trade_share * pos['cap']

    side = 'none'
    pnl = 0.0

    if rel_edge > EDGE_TRIGGER:
        # protocol overpriced: sell protocol, buy market hedge
        side = 'sell_protocol_buy_market'
        gross = (protocol_per_cap - market_per_cap) * trade_notional
        costs = (TX_COST_PCT + MISMATCH_PENALTY_PCT) * trade_notional * market_per_cap
        pnl = gross - costs
    elif rel_edge < -EDGE_TRIGGER:
        # protocol underpriced: buy protocol, sell market
        side = 'buy_protocol_sell_market'
        gross = (market_per_cap - protocol_per_cap) * trade_notional
        costs = (TX_COST_PCT + MISMATCH_PENALTY_PCT + 0.01) * trade_notional * market_per_cap
        pnl = gross - costs

    rows.append({
        'week': w,
        'imbalance': imbalance,
        'protocol_per_cap': protocol_per_cap,
        'market_per_cap': market_per_cap,
        'rel_edge': rel_edge,
        'trade_notional': trade_notional,
        'side': side,
        'agent_pnl': pnl,
    })

arb = pd.DataFrame(rows)
arb['cum_pnl'] = arb['agent_pnl'].cumsum()

print(f"Config used: alpha_rv={alpha_rv:.3f}, alpha_ivrv={alpha_ivrv:.3f}, k_lin={k_lin:.3f}, k_quad={k_quad:.3f}, util={util:.3f}")
print(f"Trades executed: {(arb['side']!='none').sum()} / {len(arb)} weeks")
print(f"Final cumulative PnL: ${arb['cum_pnl'].iloc[-1]:,.2f}")

# Imbalance regions
arb['imb_bucket'] = pd.cut(arb['imbalance'], bins=[-10,-0.10,-0.03,0.03,0.10,10], labels=['strong_supply','mild_supply','balanced','mild_demand','strong_demand'])
reg = arb.groupby('imb_bucket', dropna=False, observed=False).agg(
    weeks=('week','count'),
    trade_weeks=('side', lambda x: int((x!='none').sum())),
    mean_edge=('rel_edge','mean'),
    mean_pnl=('agent_pnl','mean'),
    total_pnl=('agent_pnl','sum')
).reset_index()

print('\nPnL by imbalance region:')
print(reg.to_string(index=False, float_format=lambda x: f'{x:,.4f}'))

fig, ax = plt.subplots(1,2,figsize=(13,4.5))
ax[0].plot(arb['week'], arb['rel_edge']*100, color='slateblue', lw=1.2)
ax[0].axhline(EDGE_TRIGGER*100, color='darkorange', ls='--', lw=1)
ax[0].axhline(-EDGE_TRIGGER*100, color='darkorange', ls='--', lw=1)
ax[0].set_title('Relative edge: protocol vs market per-cap (%)')
ax[0].set_xlabel('Week'); ax[0].set_ylabel('%')

ax[1].plot(arb['week'], arb['cum_pnl'], color='seagreen', lw=1.5)
ax[1].axhline(0, color='black', ls='--', lw=1)
ax[1].set_title('Arbitrage agent cumulative PnL (USD)')
ax[1].set_xlabel('Week'); ax[1].set_ylabel('USD')

plt.tight_layout()
out_png = os.path.join(ROOT_DATA_DIR, 'v4_arb_agent_pnl.png')
out_csv = os.path.join(ROOT_DATA_DIR, 'v4_arb_agent_timeseries.csv')
out_reg = os.path.join(ROOT_DATA_DIR, 'v4_arb_agent_region_summary.csv')
fig.savefig(out_png, dpi=150)
arb.to_csv(out_csv, index=False)
reg.to_csv(out_reg, index=False)
print(f'\nSaved: {out_png}')
print(f'Saved: {out_csv}')
print(f'Saved: {out_reg}')

Config used: alpha_rv=0.100, alpha_ivrv=0.200, k_lin=0.260, k_quad=0.100, util=0.600
Trades executed: 83 / 99 weeks
Final cumulative PnL: $476.92

PnL by imbalance region:
   imb_bucket  weeks  trade_weeks  mean_edge  mean_pnl  total_pnl
strong_supply     99           83     0.0139    4.8174   476.9221
  mild_supply      0            0        NaN       NaN     0.0000
     balanced      0            0        NaN       NaN     0.0000
  mild_demand      0            0        NaN       NaN     0.0000
strong_demand      0            0        NaN       NaN     0.0000



Saved: data/v4_arb_agent_pnl.png
Saved: data/v4_arb_agent_timeseries.csv
Saved: data/v4_arb_agent_region_summary.csv


## 2D Demand/Supply Stress Map for Arbitrage Agent

This grid stress-tests demand and supply multipliers to map where the agent has positive expected basis-arbitrage PnL on each side.

In [10]:
# 2D stress grid: demand multiplier x supply multiplier

cfg = cfg_use  # from previous cell
alpha_rv = float(cfg['alpha_rv'])
alpha_ivrv = float(cfg['alpha_ivrv'])
k_lin = float(cfg['k_lin'])
k_quad = float(cfg['k_quad'])
util = float(cfg['max_utilization'])

Ss = float(closes[-8])
rv30 = float(vol30[-8]) if np.isfinite(vol30[-8]) else 0.65
rv7 = float(vol7[-8]) if np.isfinite(vol7[-8]) else rv30
ivrv = float(np.clip(0.95 + 0.45*(rv7/max(rv30,1e-9)), 0.90, 1.45))
pos = make_pos(Ss, lev=1.0)
fv = fv_quadrature(Ss, rv30, pos['B'], pos['cap'], pos['L'], pos['pl'], pos['pu'])

base_demand = 4.0 * float(np.clip(rv7/max(rv30,1e-9), 0.6, 1.8)) * pos['cap']
base_supply = max(1.0, 2_000_000.0 * util)

D_GRID = np.linspace(0.4, 2.2, 19)  # demand multiplier
S_GRID = np.linspace(0.4, 2.2, 19)  # supply multiplier

heat_pnl = np.zeros((len(S_GRID), len(D_GRID)))
heat_side = np.zeros((len(S_GRID), len(D_GRID)))  # +1 sell protocol / -1 buy protocol / 0 no trade
heat_edge = np.zeros((len(S_GRID), len(D_GRID)))

for i, s_mult in enumerate(S_GRID):
    for j, d_mult in enumerate(D_GRID):
        demand_notional = base_demand * d_mult
        supply_notional = base_supply * s_mult
        imbalance = demand_notional / max(supply_notional,1e-9) - 1.0

        mv = vol_markup(rv30, ivrv, alpha_rv, alpha_ivrv)
        ma = amm_mult(demand_notional, supply_notional, k_lin, k_quad)
        protocol_per_cap = float((fv * mv * ma) / max(pos['cap'], 1e-9))
        market_per_cap = float(mkt_per_cap_med)
        rel_edge = (protocol_per_cap / max(market_per_cap,1e-9)) - 1.0

        trade_share = min(MAX_TRADE_SHARE, max(0.0, abs(imbalance)*2.0))
        trade_notional = trade_share * pos['cap']

        pnl = 0.0
        side = 0.0
        if rel_edge > EDGE_TRIGGER:
            gross = (protocol_per_cap - market_per_cap) * trade_notional
            costs = (TX_COST_PCT + MISMATCH_PENALTY_PCT) * trade_notional * market_per_cap
            pnl = gross - costs
            side = 1.0
        elif rel_edge < -EDGE_TRIGGER:
            gross = (market_per_cap - protocol_per_cap) * trade_notional
            costs = (TX_COST_PCT + MISMATCH_PENALTY_PCT + 0.01) * trade_notional * market_per_cap
            pnl = gross - costs
            side = -1.0

        heat_pnl[i,j] = pnl
        heat_side[i,j] = side
        heat_edge[i,j] = rel_edge

# Tabular extraction of profitable regions
rows=[]
for i, s_mult in enumerate(S_GRID):
    for j, d_mult in enumerate(D_GRID):
        rows.append({
            'supply_mult': float(s_mult),
            'demand_mult': float(d_mult),
            'imbalance': float((base_demand*d_mult)/(base_supply*s_mult)-1.0),
            'edge_pct': float(heat_edge[i,j]*100.0),
            'agent_pnl_usd': float(heat_pnl[i,j]),
            'side': 'sell_protocol_buy_market' if heat_side[i,j]>0 else ('buy_protocol_sell_market' if heat_side[i,j]<0 else 'none'),
            'profitable': bool(heat_pnl[i,j] > 0),
        })

grid_df = pd.DataFrame(rows)

summary = grid_df.groupby('side', dropna=False).agg(
    points=('side','count'),
    profitable_points=('profitable','sum'),
    mean_pnl=('agent_pnl_usd','mean'),
    max_pnl=('agent_pnl_usd','max'),
    min_pnl=('agent_pnl_usd','min')
).reset_index()

print('2D stress grid summary by arbitrage side:')
print(summary.to_string(index=False, float_format=lambda x: f'{x:,.4f}'))

best = grid_df.sort_values('agent_pnl_usd', ascending=False).head(12)
print('\nTop profitable stress points:')
print(best[['demand_mult','supply_mult','imbalance','edge_pct','agent_pnl_usd','side']].to_string(index=False, float_format=lambda x: f'{x:,.4f}'))

# Plot heatmaps
fig, ax = plt.subplots(1,3,figsize=(16,4.5))

im0 = ax[0].imshow(heat_edge*100, origin='lower', aspect='auto',
                   extent=[D_GRID.min(), D_GRID.max(), S_GRID.min(), S_GRID.max()],
                   cmap='coolwarm')
ax[0].set_title('Relative edge (%)')
ax[0].set_xlabel('Demand multiplier')
ax[0].set_ylabel('Supply multiplier')
fig.colorbar(im0, ax=ax[0], shrink=0.8)

im1 = ax[1].imshow(heat_pnl, origin='lower', aspect='auto',
                   extent=[D_GRID.min(), D_GRID.max(), S_GRID.min(), S_GRID.max()],
                   cmap='RdYlGn')
ax[1].set_title('Agent PnL per state (USD)')
ax[1].set_xlabel('Demand multiplier')
ax[1].set_ylabel('Supply multiplier')
fig.colorbar(im1, ax=ax[1], shrink=0.8)

im2 = ax[2].imshow(heat_side, origin='lower', aspect='auto',
                   extent=[D_GRID.min(), D_GRID.max(), S_GRID.min(), S_GRID.max()],
                   cmap='bwr', vmin=-1, vmax=1)
ax[2].set_title('Arbitrage side map')
ax[2].set_xlabel('Demand multiplier')
ax[2].set_ylabel('Supply multiplier')
ax[2].text(0.42, 2.08, 'blue: buy protocol / sell market', fontsize=8)
ax[2].text(1.35, 2.08, 'red: sell protocol / buy market', fontsize=8)
fig.colorbar(im2, ax=ax[2], shrink=0.8)

plt.tight_layout()
out_grid_csv = os.path.join(ROOT_DATA_DIR, 'v4_arb_agent_stress_grid.csv')
out_grid_png = os.path.join(ROOT_DATA_DIR, 'v4_arb_agent_stress_heatmaps.png')
grid_df.to_csv(out_grid_csv, index=False)
fig.savefig(out_grid_png, dpi=160)

print(f'\nSaved: {out_grid_csv}')
print(f'Saved: {out_grid_png}')

2D stress grid summary by arbitrage side:
                    side  points  profitable_points  mean_pnl  max_pnl  min_pnl
buy_protocol_sell_market     361                361    4.7980   4.7980   4.7980

Top profitable stress points:
 demand_mult  supply_mult  imbalance  edge_pct  agent_pnl_usd                     side
      2.2000       2.2000    -0.9986  -14.6172         4.7980 buy_protocol_sell_market
      0.4000       0.4000    -0.9986  -14.6172         4.7980 buy_protocol_sell_market
      0.5000       0.4000    -0.9983  -14.6172         4.7980 buy_protocol_sell_market
      0.6000       0.4000    -0.9979  -14.6172         4.7980 buy_protocol_sell_market
      0.7000       0.4000    -0.9976  -14.6172         4.7980 buy_protocol_sell_market
      0.8000       0.4000    -0.9972  -14.6172         4.7980 buy_protocol_sell_market
      0.9000       0.4000    -0.9969  -14.6172         4.7980 buy_protocol_sell_market
      1.0000       0.4000    -0.9965  -14.6172         4.7980 buy_proto


Saved: data/v4_arb_agent_stress_grid.csv
Saved: data/v4_arb_agent_stress_heatmaps.png
